In [1]:
import sys
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(level=logging.CRITICAL, force=True, stream=sys.stdout)
# logging.
n_epoch = 1000
# n_epoch = 10

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [9]:
# logging.getLogger("torch_numopt.numerical_optimizer").setLevel(logging.CRITICAL)
# logging.getLogger("torch_numopt.curvature").setLevel(logging.INFO)
logging.getLogger("torch_numopt").setLevel(logging.CRITICAL)
# logging.getLogger("torch_numopt.algorithms").setLevel(logging.DEBUG)
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

# opt = torch_numopt.GradientDescent(model.parameters())
# opt = torch_numopt.GradientDescentLipschitz(model.parameters())
# opt = torch_numopt.ConjugateGradient(model.parameters())
# opt = torch_numopt.ConjugateGradient(model.parameters(), lr_method="BB2")
# opt = torch_numopt.LBFGS(model.parameters())
# opt = torch_numopt.AdaHessian(model.parameters())
# opt = torch_numopt.DiagonalNewton(model.parameters())
# opt = torch_numopt.GaussNewton(model.parameters())
# opt = torch_numopt.Newton(model.parameters(), lr_init=1e-2)
# opt = torch_numopt.NewtonCG(model.parameters(), lr_init=1e-2)

# opt = torch_numopt.GradientDescentLS(model.parameters())
# opt = torch_numopt.ConjugateGradientLS(model.parameters())
# opt = torch_numopt.LBFGSLS(model.parameters())
# opt = torch_numopt.AdaHessianLS(model.parameters())
# opt = torch_numopt.DiagonalNewtonLS(model.parameters())
# opt = torch_numopt.GaussNewtonLS(model.parameters())
# opt = torch_numopt.NewtonLS(model.parameters())
# opt = torch_numopt.NewtonCGLS(model.parameters())

# opt = torch_numopt.GradientDescentTR(model.parameters(), radius_max=1)
# opt = torch_numopt.NewtonTR(model.parameters(), trust_region_method="cauchy")
# opt = torch_numopt.NewtonTR(model.parameters())
# opt = torch_numopt.GaussNewtonTR(model.parameters())
# opt = torch_numopt.GaussNewtonTR(model.parameters(), trust_region_method="steihaug-toint")
# opt = torch_numopt.NewtonCGTR(model.parameters())
# opt = torch_numopt.LevenbergMarquardt(model.parameters())
opt = torch_numopt.InexactLevenbergMarquardt(model.parameters())

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=n_epoch, max_patience=50)

epoch:  0, loss: 0.04471554607152939
epoch:  1, loss: 0.03438440337777138
epoch:  2, loss: 0.03426824510097504
epoch:  3, loss: 0.0331021212041378
epoch:  4, loss: 0.7273260354995728
epoch:  5, loss: 0.026669545099139214
epoch:  6, loss: 0.026669545099139214
epoch:  7, loss: 0.008644228801131248
epoch:  8, loss: 0.008644228801131248
epoch:  9, loss: 0.008644228801131248
epoch:  10, loss: 0.004915210884064436
epoch:  11, loss: 0.004915210884064436
epoch:  12, loss: 0.004852739628404379
epoch:  13, loss: 0.004852739628404379
epoch:  14, loss: 0.004817119333893061
epoch:  15, loss: 0.004817119333893061
epoch:  16, loss: 0.004783874377608299
epoch:  17, loss: 0.004783874377608299
epoch:  18, loss: 0.004751986358314753
epoch:  19, loss: 0.004751986358314753
epoch:  20, loss: 0.004719986114650965
epoch:  21, loss: 0.004719986114650965
epoch:  22, loss: 0.004687052220106125
epoch:  23, loss: 0.004687052220106125
epoch:  24, loss: 0.004655491095036268
epoch:  25, loss: 0.004655491095036268
epo

In [10]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9969322085380554
Test metrics:  R2 = 0.9900071620941162
